In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import matplotlib.pyplot as plt

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl, AugmentedPolicyControl, AugmentedFCLockedControl

In [ ]:
# 1. Setup Environment and Fleet Data
env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)
exclude_days = [1, 2, 3]

# Lock baseline spatial/temporal parameters found from Notebook 1
OPTIMAL_DP = 150.0
OPTIMAL_DT = 300

In [ ]:
# 2. Define the Module Pack Test Vector (Divisors of 16)
pack_test_values = [1, 2, 4, 8, 16]

results_data = []
print("--- RUNNING ACTUATOR CONDENSATION SENSITIVITY ---")

for pack in pack_test_values:
    print(f"\n[ Evaluating Module Pack Size: n_pack = {pack} ]")
    
    config = SimConfig(
        dP=OPTIMAL_DP,
        Dt=OPTIMAL_DT,
        N_Pd=6,
        use_smart_grid=True,
        n_pack=pack,
        apply_terminal_n_cost=False,
        apply_terminal_soc_cost=True,
        alpha_fc=4
    )
    
    benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)
    
    value_factory = build_approach(
        controller_cls=AugmentedFCLockedControl,
        plant_cls=AugmentedHybridPlant,
        solver_cls=AugmentedHybridSDPSolver,
        is_macro=False
    )
    
    report = benchmarker.run_leave_one_out(value_factory)
    avg_metrics = report.summary.loc['Average']
    
    results_data.append({
        'n_pack': pack,
        'Total Cost [$]': avg_metrics['Total Cost [$]'],
        'Operating Cost [$]': avg_metrics['Operating Cost [$]'],
        'Switching Cost [$]': avg_metrics['Switching Cost [$]'],
        'Battery Cost [$]': avg_metrics['Battery Cost [$]'],
        'Offline Compute Time [s]': avg_metrics['Offline Compute Time [s]']
    })

df_pack = pd.DataFrame(results_data).set_index('n_pack')

In [ ]:
print("\n--- ACTUATOR CONDENSATION SUMMARY TABLE ---")
print_markdown_table(df_pack)

In [ ]:
# 3. Plotting the L-Curve (Compute Time vs Total Cost)
fig, ax1 = plt.subplots(figsize=(9, 5))

times = df_pack['Offline Compute Time [s]'].values
costs = df_pack['Total Cost [$]'].values
packs = df_pack.index.values

ax1.plot(packs, costs, marker='o', linestyle='-', color='purple', linewidth=2, markersize=8, label='Total Cost [$]')
ax1.set_xlabel('Module Pack Size ($n_{pack}$)', fontsize=12)
ax1.set_ylabel('Average Total Cost [$]', fontsize=12, color='purple')
ax1.tick_params(axis='y', labelcolor='purple')
ax1.set_xticks(packs)

ax2 = ax1.twinx()
ax2.plot(packs, times, marker='s', linestyle='--', color='teal', linewidth=2, markersize=8, label='Compute Time [s]')
ax2.set_ylabel('Offline Compute Time [Seconds]', fontsize=12, color='teal')
ax2.tick_params(axis='y', labelcolor='teal')

plt.title('Actuator Condensation Tradeoff: Complexity vs. Optimality', fontsize=14, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)

os.makedirs('figures', exist_ok=True)
plt.savefig('figures/actuator_condensation_lcurve.png', dpi=300, bbox_inches='tight')
plt.show()